In [2]:
from dotenv import load_dotenv
import os
import json
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from typing_extensions import TypedDict, Annotated
from typing import List
from openai import AzureOpenAI

# Load environment variables
load_dotenv()

# Configuration
end_point = os.getenv("AZURE_OPENAI_GPT_4O_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_GPT_4O_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT_4O_API_VERSION")
deployment = os.getenv("AZURE_OPENAI_GPT_4O_DEPLOYMENT_NAME").strip()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=end_point,
    api_key=api_key
)

# Configuration
end_point = os.getenv("AZURE_OPENAI_GPT_4O_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_GPT_4O_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT_4O_API_VERSION")
deployment = os.getenv("AZURE_OPENAI_GPT_4O_DEPLOYMENT_NAME").strip()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=end_point,
    api_key=api_key
)


class MessageState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
 

def get_person_details(person_name: str)-> str:
    return f"{person_name} is working as data scientist"



def get_person_location(person_name: str) -> str:
    return f"{person_name} lives in Bangalore."

def dynamic_router_node(state: MessageState) -> MessageState:
    user_message = state["messages"][-1]

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": user_message.content}],
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "get_person_details",
                    "description": "Get information about a person",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "person_name": {"type": "string", "description": "Name of the person"}
                        },
                        "required": ["person_name"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "get_person_location",
                    "description": "Get location of a person",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "person_name": {"type": "string", "description": "location  of the person"}
                        },
                        "required": ["person_name"]
                    }
                }
            }
        ],
        tool_choice="auto",  # ✅ LLM decides automatically
    )

    choice = response.choices[0]

    # If model decides to call a tool
    if choice.finish_reason == "tool_calls":
        tool_call = choice.message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "get_person_details":
            result = get_person_details(**args)
        elif tool_call.function.name == "get_person_location":
            result = get_person_location(**args)
        return {"messages": [HumanMessage(content=result)]}

    # Otherwise normal LLM response
    reply = choice.message.content
    return {"messages": [HumanMessage(content=reply)]}


builder = StateGraph(MessageState)
builder.add_node("dynamic_router_node", dynamic_router_node)
builder.add_edge(START, "dynamic_router_node")
builder.add_edge("dynamic_router_node", END)
graph = builder.compile()


print("\n=== Test Case 1: Person Details Tool ===")
messages = graph.invoke({"messages": [HumanMessage(content="Who is Alice?")]})
for msg in messages["messages"]:
    msg.pretty_print()

print("\n=== Test Case 2: Person Location Tool ===")
messages = graph.invoke({"messages": [HumanMessage(content="Where does Alice live?")]})
for msg in messages["messages"]:
    msg.pretty_print()


print("\n=== Test Case 3: Normal LLM Response ===")
messages = graph.invoke({"messages": [HumanMessage(content="Tell me a joke")]})
for msg in messages["messages"]:
    msg.pretty_print()
 


=== Test Case 1: Person Details Tool ===
================================ Human Message =================================

Who is Alice?
================================ Human Message =================================

Alice is working as data scientist

=== Test Case 2: Person Location Tool ===
================================ Human Message =================================

Where does Alice live?
================================ Human Message =================================

Alice lives in Bangalore.

=== Test Case 3: Normal LLM Response ===
================================ Human Message =================================

Tell me a joke
================================ Human Message =================================

Sure! Here's one:

Why don't skeletons fight each other?

Because they don't have the guts! 🤣
